# 05. 관측 전처리와 행동 계약 / 60분

**강사 설명 → 완성된 셀 실행 → 조건 비교 → 결과 해설** 순서입니다. 코드 작성 과제나 관찰 답안 제출은 없습니다.

같은 숫자 개수라도 축·이름·단위가 다르면 다른 입력입니다. 상태8과 행동7을 혼동하지 않습니다.

[단원 실행 안내](README.md) · [슬라이드 실습 시작](https://docs.google.com/presentation/d/1yAFoaiH15_qTveoSi5r1gF1TPE7bFf2CrLVLUolsg4k/edit#slide=id.d5_lab05_start)

각 단원은 새 커널에서 시작합니다. 노트북 셀은 위에서 아래로 실행하고, `bash` 명령은 지정한 터미널에서 실행합니다. 결과가 이미 있으면 재사용 여부와 실행 이름을 확인합니다.


## 1. 아키텍처 — 지금 실행하는 부분

![관측 전처리와 행동 계약의 입력·처리·출력](../교재그림/guided-vla/day5-05.svg)

원본 NPZ의 RGB 영상을 두 축 반전한 뒤 HWC를 CHW로 바꾸고 255로 나눕니다. make_batch는 위치3·axis-angle3·그리퍼2를 상태8로 만들고, 제공 preprocessor가 카메라 이름과 정규화를 맞춥니다.

**그림을 읽는 순서:** 1→2→3 뒤 아래 4→5→6으로 이어집니다. 실시간 반복은 새 관측이 다음 입력으로 들어오는 과정이며, 저장 결과를 읽는 셀은 모델을 다시 실행하지 않습니다.


### 전체 실행 지도에서 현재 단원 찾기

![5일차 실행 경계 전체 지도](../교재그림/guided-vla/detailed/day5-system.svg)

브라우저는 실행을 요청하고 결과를 표시합니다. 커널이 호출한 별도 runner가 물리 환경과 정책을 연결합니다. 01·02의 합성 셀은 구조 설명용이고 실제 환경 실행은 04, 모델 실행은 06부터입니다. 아래 단원별 도식에서 이번에 실제로 실행할 범위를 확인합니다.

## 상세 해설 — 앞선 수업 없이도 흐름을 읽기

아래 설명은 1~3일차에서 배운 개념도 이번 실습의 입력·코드·출력에 맞춰 다시 풉니다. 실행 중에는 필요한 부분을 읽고, 수업 뒤에는 독립적인 참고 자료로 사용할 수 있습니다. 추가 작성 과제는 없습니다.

### 1. 기초 복습: 축·자료형·값 범위

- HWC=(높이,너비,색); CHW=(색,높이,너비)
- RGB uint8은 0~255, float 영상은 0~1로 변환
- 축 순서 변경은 영상 회전·채널 순서 변경과 다릅니다.
- 상태8과 행동7은 다른 의미와 구성입니다.

영상 배열의 shape만 보고 의미를 추정하지 말고 각 축 이름을 붙입니다. raw.shape=(256,256,3)은 HWC이고 converted.shape=(3,256,256)은 CHW입니다. transpose나 permute는 차원의 배치를 바꾸며 좌우 반전이나 RGB↔BGR 변환을 대신하지 않습니다. 서로 다른 연산을 한 줄에 연결했더라도 역할은 나눠 이해합니다.

uint8은 0~255 정수 범위입니다. float로 바꾸고 255로 나누면 0~1 범위가 됩니다. 자료형을 바꾸는 것과 숫자의 크기를 바꾸는 것은 별도 연산입니다. 단순 float 변환만 하면 범위는 여전히 0~255입니다. 뒤의 모델 정규화와도 구별합니다.

상태8은 끝단 위치3, quaternion에서 변환한 axis-angle 회전3, 그리퍼 관절2입니다. 행동7은 제어 입력의 이동3·회전3·그리퍼1입니다. 관측이 8차원인데 출력이 7차원인 것이 오류는 아닙니다. 입력 상태와 행동 표현이 서로 다른 계약이기 때문입니다.


### 2. 실행 아키텍처: 관측 계약을 정책 계약으로

![실행 주체와 데이터 경계](../교재그림/guided-vla/detailed/day5-05.svg)

- obs[pixels] + obs[robot_state] + task
- make_batch: 방향·축·dtype·상태 회전 표현 변환
- preprocessor: 제공된 이름 매핑·정규화 등 적용
- 정책에 전달; 출력은 postprocessor로 복원

make_batch는 환경이 반환한 자료 구조를 정책 처리 경로가 기대하는 key로 바꿉니다. 영상은 observation.images.image와 image2, 상태는 observation.state, 언어는 task로 묶습니다. 카메라가 두 개라는 숫자만 맞추는 것이 아니라 각 이름이 학습 시 의미와 연결되도록 제공 processor를 사용합니다.

원본 환경의 회전은 quaternion이며 make_batch는 LiberoProcessorStep의 변환으로 axis-angle3을 만듭니다. quaternion의 네 값을 그대로 상태에 붙이거나 단순히 하나를 삭제하면 같은 회전 표현이 아닙니다. 위치3+변환 회전3+그리퍼2를 torch.cat으로 연결하고 shape와 유한성을 검사합니다.

make_batch 뒤에는 모델과 함께 배포된 preprocessor가 더 적용됩니다. 노트북의 영상 변환 셀만으로 전체 정책 전처리가 완료되었다고 설명하지 않습니다. 반대로 postprocessor는 예측한 행동을 환경에 넘길 표현으로 복원하는 경계이며 화면 표시 기능이 아닙니다.


### 3. 코드 추적: 영상 변환 한 줄을 분해하기

- raw[::-1,::-1]: 높이·너비 방향을 함께 반전
- copy(): 연속 버퍼를 만들어 음수 stride 경계 처리
- permute/transpose: HWC → CHW
- float()/255: dtype와 범위를 맞춘 뒤 pre 호출

[::-1,::-1]은 첫 두 축을 역순으로 읽는 NumPy view를 만들 수 있습니다. 방향을 맞추는 목적과 이후 tensor 변환의 메모리 조건을 함께 고려해 copy()를 둡니다. torch.from_numpy가 음수 stride 배열을 그대로 받지 못하는 경계를 피하고 원본과 독립된 입력을 만듭니다.

그 뒤 permute(2,0,1)는 색 축을 앞으로 옮깁니다. 표시할 때 matplotlib는 HWC를 기대하므로 converted.transpose(1,2,0)으로 되돌려 보여 줍니다. 표시를 위해 되돌린 배열을 모델 입력에 다시 넣거나 추가 방향 반전을 넣지 않습니다. 화면용 변환과 계산용 변환을 분리합니다.

make_batch는 각 영상이 정확히 (256,256,3), uint8인지 검사하고 상태가 유한한 8차원인지 확인합니다. NaN은 숫자처럼 배열 안에 존재하지만 모델 계산을 오염시킬 수 있어 isfinite 검사가 필요합니다. shape가 맞다는 것만으로 올바른 입력은 아닙니다.


### 4. 입력 수정 시 같이 확인할 항목

- 해상도·카메라 이름·RGB 순서·방향을 각각 확인합니다.
- 상태 통계는 실제 8차원이며 오래된 shape=6을 따르지 않습니다.
- 카메라 누락을 가짜 이미지로 조용히 채우지 않습니다.
- raw/converted 표는 변환 확인이며 성능 평가가 아닙니다.

준비 모델의 오래된 config에는 상태 shape=6 메타데이터가 있을 수 있지만 수업의 실제 상태와 정규화 통계는 8입니다. load_policy는 통계를 확인하고 예상 범위의 메타데이터만 받아들입니다. 숫자 6에 맞춰 상태 두 값을 자르는 수정은 그리퍼 정보를 잃고 학습 입력과 달라지게 합니다.

새 카메라나 다른 해상도를 연결한다면 make_batch 검사만 느슨하게 하지 말고 모델의 입력 이름·학습 시점·전처리 설정까지 확인해야 합니다. 필요한 camera2가 없을 때 camera1을 복제하거나 임의 camera3를 넣는 것은 수치 형상만 맞춘 다른 입력입니다.

자동 표에서 shape·dtype·min/max가 예상과 맞는지 확인한 뒤 두 영상의 방향과 색을 함께 봅니다. min/max가 0~1이라고 카메라 의미가 올바르다는 보장은 없습니다. 강사는 수치 계약과 시각 의미를 함께 설명하고 다음 추론 단계로 연결합니다.


## 2. 코드 위치와 변경 범위

| 구분 | 이 단원에서 확인할 내용 |
|---|---|
| 실제 소스 | [scripts/course_runtime.py : make_batch](scripts/course_runtime.py) |
| 바꾸어 볼 조건 | `표시용 raw / converted 비교` |
| 예상되는 차이 | 원본/변환 영상과 shape·dtype·최솟값·최댓값을 함께 확인합니다. 배열의 축을 바꾼 뒤 표시할 때는 HWC로 돌립니다. |
| 직접 구현·수정할 때 | config의 오래된 shape=6을 따라 상태를 자르지 않습니다. 필수 두 카메라를 유지하고 임의의 가짜 camera3를 추가하지 않습니다. 행동7은 절대 위치가 아닙니다. |

설정값 변경은 아래 준비된 비교 실행으로 진행합니다. 소스를 직접 변경하는 경우에는 실행 중인 프로세스를 종료하고 다시 실행해야 합니다. ROS 패키지 파일은 재빌드·환경 source까지 확인합니다.


### 실제 코드 읽기 — 입력·처리·출력 연결

원본 NPZ의 RGB 영상을 두 축 반전한 뒤 HWC를 CHW로 바꾸고 255로 나눕니다. make_batch는 위치3·axis-angle3·그리퍼2를 상태8로 만들고, 제공 preprocessor가 카메라 이름과 정규화를 맞춥니다.


In [ ]:
from pathlib import Path
from IPython.display import Code, display
import ast
source_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'10.VLA_v2/scripts').is_dir())
source_path = source_root/'10.VLA_v2'/'scripts/course_runtime.py'
text = source_path.read_text()
# 함수 위치를 AST로 찾아 해당 함수 전체를 표시합니다. 파일을 import하거나 모델을 로드하지 않습니다.
functions = [node for node in ast.parse(text).body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))]
for node in functions:
    if node.name in ['make_batch']:
        print(source_path, '함수:', node.name, '줄:', node.lineno)
        display(Code('\n'.join(text.splitlines()[node.lineno-1:node.end_lineno]), language='python'))


## 05의 화면: 저장 영상 확인

05는 저장된 관측 배열을 분석하므로 새 MuJoCo GUI를 열지 않습니다. 배열 읽기 셀에서 같은 실행의 두 PNG를 표시합니다. 결과가 없을 때의 inspect 생성도 이 단원에서는 창 없이 수행합니다. 실제 추론·실행 GUI는 06·07에서 사용합니다.


## 전체 프로세스에서 지금 어디에 있나요?

![가상 환경의 관측에서 VLA 입력·행동·재관측까지의 흐름](../교재그림/vla-observation-action-flow.svg)

그림의 **파란 단계는 05에서 직접 실행** , 보라색은 실제 runner의 입력 준비 중 설명으로 살펴보는 부분, 초록색·주황색은 06·07에서 이어지는 부분입니다. 05의 영상 변환은 저장된 영상 한 장으로 같은 연산을 확인하는 예제이며, 전체 정책 입력을 완성하는 코드는 아닙니다.

1. **04 — 환경 초기화:** LIBERO의 과제와 초기 배치를 정하고 reset합니다. 두 영상·로봇 상태·언어 지시를 얻습니다.
2. **05 — 저장 배열 확인:** `initial-observation.npz`에서 두 영상과 상태 8개를 읽습니다. 언어 지시는 별도로 `result.json`의 `task`에 기록됩니다.
3. **05 — 영상 변환 실습:** 방향을 맞추고, 배열 축과 자료형·값 범위를 바꿉니다. 실제 `make_batch`는 두 카메라 모두에 이 처리를 적용합니다.
4. **실제 runner의 나머지 입력 준비:** 저장된 processor가 카메라 이름을 바꾸고 배치 축을 추가하며, 지시를 토큰으로 변환하고 CUDA로 옮긴 뒤 상태를 정규화합니다. 05에서는 코드와 설정을 읽어 역할을 이해합니다.
5. **06 — 정책 추론과 후처리:** 준비한 입력에서 행동 묶음을 예측하고 학습 정규화를 되돌립니다. 한 번의 추론 결과를 확인하며 환경에 행동을 적용하지 않습니다.
6. **07 — 실행과 재관측:** 기본적으로 앞 10개의 행동을 하나씩 적용하고, 계속 진행할 경우 새 관측으로 다시 추론합니다. 각 step에서 성공·종료를 확인합니다.

**파일 연결과 실시간 연결은 구분합니다.** 05는 04의 저장 관측을 읽는 연습입니다. 실제 06·07 runner는 자신의 환경을 reset한 관측으로 입력을 만들며, 매번 04의 PNG를 읽어 추론하지 않습니다. 07의 다음 묶음은 행동 후 새 관측을 사용합니다.


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [5일차 실행 안내](README.md)를 참고합니다.

**5일차 환경:** `nepes-course:physical-ai-vla-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 같은 날에는 준비된 개인 VLA 환경을 유지합니다.

**실행 전 확인 — 컨테이너 내부 Terminal:**

```bash
cd /workspace
/opt/course/bin/python 10.VLA_v2/scripts/preflight.py
```

Python 커널도 `/opt/course/bin/python`을 사용합니다. 강사 배포 `Data/models/smolvla-libero-probe`(LIBERO용 정책 revision `31d453f7edd78c839a8bbc39744a292686daf0de`), `Data/cache/smolvla-probe/huggingface`, `Data/cache/smolvla-probe/libero`와 LIBERO 자산이 필요합니다. 파일 누락은 호스트 개인 Data의 마운트·배포본을 확인합니다. 수강생이 `pip install`, Docker build, 모델 다운로드를 하지 않습니다. CUDA/EGL 문제는 전용 이미지·NVIDIA runtime과 다른 GPU 작업을 확인합니다.

**실행과 저장:** 04부터 08까지의 노트북의 `SESSION = 'v2-session-01'`은 같은 회차이며 결과는 `Data/logs/day5/v2-session-01/<실행이름>/`에 남습니다. 새 비교 전체는 모든 후속 노트북의 SESSION을 함께 `session-02` 등으로 바꿉니다. `run_lesson`은 동일 조건의 `completed` 결과를 읽고 없을 때 실행합니다. 미완료·조건 불일치는 기존 파일을 삭제하거나 성공값으로 바꾸지 말고 새 회차/실행 이름을 사용합니다. JSON은 있는데 필요한 PNG/NPZ/NPY/영상이 없으면 부분 손상으로 보고 새 회차에서 다시 생성합니다.

**한 번에 한 정책 프로세스만 실행합니다.** 이전 실행이 끝난 뒤 다음 셀을 실행하고 다른 노트북의 infer/rollout을 동시에 시작하지 않습니다. 실행 중단 후에는 해당 Terminal이나 실행 로그에서 자식 프로세스까지 종료됐는지 확인합니다. `status=completed`는 절차 완료, rollout의 `success=true`는 환경 과제 성공이며 inspect/infer의 `success=false`는 아직 작업을 하지 않았다는 뜻입니다.


## 이전 자료의 위치·필요한 내용·없는 파일 복구

[04_environment.ipynb의 「예제」](04_environment.ipynb)가 만든 `Data/logs/day5/<SESSION>/inspect-01/initial-observation.npz`를 읽습니다. 그 파일에는 `image,image2` 두 RGB uint8 `(256,256,3)` 배열과 `state` float `(8,)`가 있습니다. 상태의 앞3개는 끝단 위치, 다음3개는 quaternion을 axis-angle로 바꾼 회전, 마지막2개는 그리퍼 관절입니다.

아래 첫 설정 셀에서 SESSION/RUNS/run_lesson을 다시 만들고, 배열 읽기 셀은 `run_lesson('inspect','inspect-01')`로 동일 조건 결과를 확인하거나 **없으면 정책 없이 reset하여 생성** 합니다. 04 노트북을 다시 열 필요가 없습니다. JSON만 있고 배열이 없거나 조건이 다르면 시작 절의 새 SESSION 복구를 따릅니다.

상세 변환 위치는 [scripts/course_runtime.py의 make_batch](scripts/course_runtime.py)와 `/workspace/Data/models/smolvla-libero-probe/policy_preprocessor.json`입니다. 핵심은 raw 영상을 가로·세로 한 번 반전 → HWC에서 CHW → float32/255, image→camera1·image2→camera2 이름 변경, 상태8에 저장 정규화 통계 적용입니다. 아래 셀의 변환은 raw NPZ에서 시작하고 표시용 PNG에 다시 반전을 적용하지 않습니다.


## 개념

상태8은 끝단 위치3 + axis-angle 회전3 + 그리퍼 관절2입니다. config의 오래된 상태 shape=6과 달리 실제 데이터·정규화 통계는 8입니다. 자르거나 가짜 값을 넣지 않습니다. 코드가 통계 차원을 검사합니다.

두 RGB 영상을 가로·세로 반전하고 HWC uint8→CHW float `[0,1]`로 바꿉니다. 제공 processor가 image/image2를 camera1/camera2로 변경하고 상태를 정규화합니다. camera3는 기존 모델의 누락 처리 경로를 사용하며 필수 영상 둘이 없으면 오류입니다.

행동7은 정규화된 끝단 이동3·회전3·그리퍼1입니다. 절대 좌표나 거북이 속도가 아닙니다. 제공 postprocessor가 학습 정규화를 되돌리고 환경 제어기로 전달합니다.

### 단계별 활동

진행: 저장 배열 읽기15분 → RGB/축/형상 확인15분 → 상태 구성15분 → 정규화·행동 계약15분. make_batch 함수와 policy_preprocessor.json을 함께 읽어 각 변환 위치를 찾으세요. 배열의 shape와 값 범위는 서로 다른 정보입니다. 위치3과 회전3을 서로 교환하면 차원 검사는 통과해도 의미가 틀린다는 점을 토의합니다.


In [ ]:
# 제공된 실행 셀입니다. 출력과 파일을 확인하며 위에서 아래로 실행합니다.
from pathlib import Path
import sys, subprocess, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '10.VLA_v2').is_dir())
SCRIPTS = ROOT / '10.VLA_v2/scripts'
sys.path.insert(0, str(SCRIPTS))
from course_runtime import DATA, lesson_runs, run_or_read_lesson
# 04~08 모두 같은 회차를 사용합니다. 전체 비교를 새로 할 때 session-02 등으로 함께 변경하세요.
SESSION = 'v2-session-01'
RUNS = lesson_runs(DATA, SESSION)
RUNS.mkdir(parents=True, exist_ok=True)
def run_lesson(mode, name, seed=7, execute=10, steps=280, initial_state=0):
    return run_or_read_lesson(SCRIPTS, RUNS, mode, name, seed, execute, steps, initial_state)


## 예제


## 첫 배열 읽기 셀: 무엇을 확인하나요?

`inspection = run_lesson('inspect', 'inspect-01')`은 현재 SESSION의 기준 결과를 확인합니다. 같은 조건의 완료 결과가 있으면 읽고, 없다면 정책을 로드하지 않고 생성합니다. `obs = np.load(...)`는 그 실행에서 저장한 숫자 배열을 엽니다.

| 항목 | 저장된 형태 | 뜻 | 확인 이유 |
|---|---|---|---|
| `image` | `(256, 256, 3)`, `uint8` | 첫 카메라의 RGB 영상 | 높이·너비·색상 채널과 자료형이 맞는지 |
| `image2` | `(256, 256, 3)`, `uint8` | 다른 시점의 RGB 영상 | 두 번째 필수 관측이 실제로 있는지 |
| `state` | `(8,)`, 실수 배열 | 같은 시점의 로봇 상태 | 값의 개수와 유한한 수인지 |
| `inspection['task']` | 문자열 | 어떤 작업을 수행할지 나타내는 지시 | 동일 영상에서도 목표에 따라 행동이 달라지므로 |

`assert`는 예상 조건을 검사합니다. 통과했다는 것은 해당 형상·자료형·값 조건이 맞다는 뜻이며, 사진의 내용과 카메라 역할까지 자동으로 검증했다는 뜻은 아닙니다.

### 상태 8개를 읽는 법

```text
state = [ x, y, z,  rx, ry, rz,  q_left, q_right ]
          위치 3      회전 3        그리퍼 관절 2
```

- `state[:3]`: 로봇 끝단이 현재 어디에 있는지 나타냅니다.
- `state[3:6]`: 현재 끝단의 방향을 axis-angle 형식의 3개 값으로 나타냅니다. 각각을 단순한 roll·pitch·yaw 각도로 해석하지 않습니다.
- `state[6:]`: 두 그리퍼 관절의 현재 위치입니다. 하나의 열기/닫기 명령과는 다릅니다.

환경의 원래 회전은 quaternion입니다. 수업의 `make_batch`가 axis-angle로 변환한 뒤 위치·그리퍼 관절과 이어 붙입니다. **저장된 NPZ의 상태는 이미 이 변환을 거친 8개 값** 이므로 여기서 quaternion 변환을 다시 하지 않습니다. 다만 정책용 평균·표준편차 정규화는 아직 적용하지 않은 값입니다.


In [ ]:
# 제공된 실행 셀입니다. 출력과 파일을 확인하며 위에서 아래로 실행합니다.
import numpy as np
# 04의 결과가 없으면 여기서 정책 로드 없이 reset하고 같은 입력 파일을 생성합니다.
# inspect는 초기 관측만 저장합니다. 일치하는 완료 기록이 있으면 새 실행 없이 재사용합니다.
inspection = run_lesson('inspect', 'inspect-01')
# 저장 배열을 읽는 단계입니다. 이 호출 자체가 모델 추론이나 새 환경 실행을 시작하지 않습니다.
obs = np.load(RUNS/'inspect-01/initial-observation.npz')
print({key: obs[key].shape for key in obs.files})
assert obs['state'].shape == (8,)
for camera in ('image', 'image2'):
    assert obs[camera].shape == (256,256,3)
    assert obs[camera].dtype == np.uint8
assert np.isfinite(obs['state']).all()
print('위치:', obs['state'][:3], '회전:', obs['state'][3:6], '그리퍼:', obs['state'][6:])
from IPython.display import Image, display
for camera in ('image', 'image2'):
    display(Image(filename=str(RUNS/'inspect-01'/f'{camera}.png')))


### 입력과 실행 흐름 확인


## 영상 변환 셀: 한 줄을 네 연산으로 나누어 읽기

<!-- procedure:lesson-41-3ed9da:start -->
![저장 관측을 모델 입력으로 읽기](../교재그림/vla-observation-action-flow.svg)

05는 저장 배열의 변환을 이해합니다. 한 장 변환만으로 실제 정책 입력 전체를 구성한 것은 아닙니다.
<!-- procedure:lesson-41-3ed9da:end -->


아래 셀은 `obs['image']` 한 장을 사용합니다. 현재 출력문은 원본 형상·자료형, 변환 후 형상·최솟값·최댓값을 보여 줍니다.

| 연산 | 하는 일 | 하지 않는 일 |
|---|---|---|
| `raw[::-1, ::-1].copy()` | 높이·너비 방향을 모두 뒤집어 LIBERO 어댑터의 방향에 맞춤 | RGB 색상 순서는 바꾸지 않음 |
| `.transpose(2, 0, 1)` | HWC → CHW로 배열 축 순서를 변경 | 사진을 회전하거나 확대하지 않음 |
| `.astype('float32')` | 정수 픽셀을 실수로 표현 | 이 단계만으로 0–1 범위가 되지는 않음 |
| `/ 255` | 0–255 범위를 0–1 범위로 변환 | 데이터셋 평균·표준편차 정규화와는 다른 연산 |

가령 **설명용 픽셀** `[0, 128, 255]`는 나누기 후 약 `[0.0, 0.502, 1.0]`이 됩니다. 실제 영상에서 이 값이 관측됐다는 뜻은 아닙니다. 영상에 완전한 검정·흰색이 없으면 출력 최솟값·최댓값이 정확히 0과 1일 필요도 없습니다.

`(256, 256, 3)`은 256×256개의 위치마다 RGB 값 3개가 있다는 뜻입니다. `(3, 256, 256)`은 같은 값을 색상 채널별 평면으로 묶었다는 뜻입니다. 정보량은 그대로이며 축 배치만 달라집니다.

**원본 NPZ에서 한 번만 변환합니다.** 04에서 본 PNG는 이미 표시 방향을 맞춰 저장한 그림입니다. 그 PNG에 같은 반전을 다시 적용하면 방향이 되돌아갑니다. 이 셀의 `converted`는 확인용 NumPy 배열이며 실제 모델 추론을 호출하지 않습니다.


In [ ]:
# 제공된 실행 셀입니다. 출력과 파일을 확인하며 위에서 아래로 실행합니다.
raw = obs['image']
converted = raw[::-1, ::-1].copy().transpose(2,0,1).astype('float32')/255
print(raw.shape, raw.dtype, converted.shape, converted.min(), converted.max())
assert converted.shape == (3,256,256)


## 실제 정책 입력에는 무엇이 더 필요한가요?

실제 runner는 `pre(make_batch(obs, task))`를 호출합니다. `make_batch`가 두 영상·상태 벡터·지시를 모으고, 배포된 `policy_preprocessor.json`의 처리 순서가 이어집니다.

| 처리 | 이번 실행에서의 역할 |
|---|---|
| 카메라 이름 매핑 | `image` → `camera1`, `image2` → `camera2`; 학습 때의 입력 이름과 대응 |
| 배치 축 추가 | 한 관측도 여러 관측을 처리하는 인터페이스에 맞춰 묶음 크기 1로 표현 |
| 언어 지시 처리 | 문자열을 tokenizer로 토큰 ID와 마스크로 바꾸어 모델에 전달 |
| 장치 이동 | 텐서가 추론에 사용하는 CUDA 장치에 놓이도록 준비 |
| 상태 정규화 | 학습 자료의 저장된 평균·표준편차로 상태 8개 각각의 척도를 맞춤 |

상태 정규화는 개념적으로 `(현재 값 - 학습 평균) / 학습 표준편차`입니다. 현재 샘플 8개의 평균을 임의로 빼거나, 모든 성분을 같은 단위로 취급하는 방식이 아닙니다. 이번 processor 설정에서 영상 정규화 방식은 `IDENTITY`이므로 이미 255로 나눈 영상에 임의의 평균·표준편차 처리를 추가하지 않습니다.

설정 일부에 남은 상태 길이 6이라는 메타데이터만 보고 실제 값을 자르지 않습니다. 수업 어댑터는 실제 상태와 배포된 정규화 통계가 8개인지 검사합니다. 이 설명은 **이번 체크포인트와 수업 코드의 조합** 에 대한 것이며 모든 VLA가 상태 8개를 사용한다는 뜻은 아닙니다.

## 출력 행동 7개는 상태 8개와 어떻게 다른가요?

| 구분 | 상태 입력 | 행동 출력 |
|---|---|---|
| 질문 | 지금 어떤 자세인가? | 다음에 어떻게 움직일 것인가? |
| 구성 | 위치 3 + 회전 3 + 그리퍼 관절 2 | 끝단 이동 3 + 회전 3 + 그리퍼 제어 1 |
| 역할 | 현재 상태를 정책에 알려 줌 | 환경의 제어기에 전달할 명령 |
| 주의 | 실제 관측 순서와 단위를 보존 | 절대 목표 좌표나 로봇의 관절각 7개로 해석하지 않음 |

그리퍼는 현재 두 관절의 상태를 관측하면서도, 동작은 하나의 열기/닫기 제어값으로 명령할 수 있습니다. 따라서 **입력 상태와 출력 행동의 길이가 같을 필요가 없습니다.** 정책은 상태 벡터에서 숫자 하나를 지워 행동을 만드는 것이 아니라, 영상·지시·상태를 함께 사용해 행동을 예측합니다.

06의 실제 경로는 `policy.predict_action_chunk(batch)` 다음에 `post(...)`를 적용합니다. 후처리는 학습 정규화를 되돌려 환경이 사용할 행동 값으로 복원합니다. 저장되는 묶음은 이 후처리를 거친 `50×7` 배열입니다. 07은 각 행동의 값을 `[-1, 1]`로 제한해 환경의 제어기에 전달합니다. **학습 정규화를 되돌린 값도 환경 제어기의 정규화된 명령일 수 있으므로** , 숫자 0.1을 곧바로 0.1m 이동으로 읽지 않습니다.

행동이 적용되면 물리 상태와 카메라 영상이 바뀝니다. 계속 진행할 때 이 새 관측으로 다시 입력을 구성하는 것이 폐루프입니다.


### 코드 해설 — 입력부터 결과까지

**8과 7을 분해하기:** 관측 상태8은 위치3+회전3+그리퍼 관절2입니다. 행동7은 이동3+회전3+그리퍼1이므로 같은 구성이나 단위가 아닙니다. 상태를 6으로 자르면 손가락 관절 정보가 사라지고 저장된 정규화 통계와도 맞지 않습니다.

영상 `(256,256,3)` uint8을 `(3,256,256)` float로 바꾸고 255로 나누면 값 범위가 0–1입니다. 축 변경·방향 반전·정규화는 각각 다른 연산입니다. 전처리를 두 번 적용하지 않도록 실제 processor의 책임을 확인합니다.

**확인 질문:** 두 camera의 shape가 같더라도 이름을 서로 바꾸면 왜 정책 입력의 의미가 달라지나요?

**적용·재확인:** 자동 출력과 실제 파일의 조건을 함께 확인합니다. 완료된 동일 조건 결과는 다시 읽고, 새 실험은 새 실행 이름을 사용합니다. `SESSION`을 바꾸면 후속 노트북도 맞춥니다. 형상 검사·실행 완료·작업 성공을 따로 확인하며, 예시 수치를 자기 측정값으로 복사하지 않습니다.


## 실행 실습 — 표시용 raw / converted 비교

원본/변환 영상과 shape·dtype·최솟값·최댓값을 함께 확인합니다. 배열의 축을 바꾼 뒤 표시할 때는 HWC로 돌립니다.


In [ ]:
import matplotlib.pyplot as plt
# converted는 CHW이므로 표시할 때만 HWC로 되돌립니다. 모델 입력을 다시 반전하지 않습니다.
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(raw); axes[0].set_title('Raw HWC uint8')
axes[1].imshow(converted.transpose(1, 2, 0)); axes[1].set_title('Oriented CHW float → display HWC')
for ax in axes: ax.axis('off')
plt.show()
print('원본:', raw.shape, raw.dtype, '범위:', int(raw.min()), int(raw.max()))
print('변환:', converted.shape, converted.dtype, '범위:', float(converted.min()), float(converted.max()))
assert np.allclose(converted.transpose(1,2,0), raw[::-1,::-1]/255)


## 결과 해설과 다음 단계

같은 숫자 개수라도 축·이름·단위가 다르면 다른 입력입니다. 상태8과 행동7을 혼동하지 않습니다.

config의 오래된 shape=6을 따라 상태를 자르지 않습니다. 필수 두 카메라를 유지하고 임의의 가짜 camera3를 추가하지 않습니다. 행동7은 절대 위치가 아닙니다.

입력 계약을 맞췄으므로 이제 정책이 한 번에 예측한 행동 묶음을 읽습니다.

출력 표·그래프·영상과 강사 설명을 함께 확인합니다. 별도 관찰 답안을 작성하지 않습니다.
